In [4]:
import os
import sys

sys.path.insert(0, os.path.abspath(".."))

from datetime import datetime, timedelta
from random import randint, random

from engines import GAMealPlanner, get_pantry_ingredient, load_all_ingredients, make_preferences
from models import (
    MealPlanningEnvironment,
    Pantry,
)

In [5]:
preferences = make_preferences()

In [6]:
all_ingredients = load_all_ingredients("../recipe_extraction/supplemented_structured_ingredients.json")

In [7]:
CURRENT_DATE = datetime.now()

In [8]:
pantry_ingredient_name_by_quantity = {
    "chicken breast": 800,
    "broccoli": 1500,
    "rice": 1000,
}

In [9]:
pantry_ingredients = [
    get_pantry_ingredient(name, CURRENT_DATE + timedelta(days=randint(1, 7)), random(), all_ingredients)
    for name in pantry_ingredient_name_by_quantity.keys()
]

In [10]:
pantry_ingredient_by_quantity = dict(zip(pantry_ingredients, pantry_ingredient_name_by_quantity.values()))

In [11]:
pantry = Pantry()

for pantry_ingredient, quantity in pantry_ingredient_by_quantity.items():
    pantry.add(
        pantry_ingredient,
        quantity,
    )

In [12]:
pantry.print()

---
Quantity: 800 g
Ingredient: chicken breast
	Estimated Expiration Date: 2026-04-28
	Estimated Financial Cost per 100g: EUR 0.62
	Nutritional Information:
		Calories: 125.0 kcal
		Carbohydrates: 1.79 g
		Sugar: 0.0 g
		Protein: 16.07 g
		Fat: 5.36 g
		Saturated Fat: 1.79 g
		Fiber: 1.8 g
		Sodium: 571.0 mg
		Gluten Free: Yes
		Lactose Free: Yes
		Vegetarian: No
		Vegan: No
---
---
Quantity: 1500 g
Ingredient: broccoli
	Estimated Expiration Date: 2026-04-30
	Estimated Financial Cost per 100g: EUR 0.39
	Nutritional Information:
		Calories: 31.0 kcal
		Carbohydrates: 6.27 g
		Sugar: 1.4 g
		Protein: 2.57 g
		Fat: 0.34 g
		Saturated Fat: 0.039 g
		Fiber: 2.4 g
		Sodium: 36.0 mg
		Gluten Free: Yes
		Lactose Free: Yes
		Vegetarian: Yes
		Vegan: Yes
---
---
Quantity: 1000 g
Ingredient: rice
	Estimated Expiration Date: 2026-05-01
	Estimated Financial Cost per 100g: EUR 0.55
	Nutritional Information:
		Calories: 356.0 kcal
		Carbohydrates: 80.0 g
		Sugar: 0.0 g
		Protein: 6.67 g
		Fat: 0.0 g


In [13]:
meal_planning_environment = MealPlanningEnvironment(
    recipes=[],
    pantry=pantry,
    preferences=preferences,
)

In [14]:
meal_planning_environment.load_recipes_from_json("../recipe_extraction/supplemented_structured_recipes.json")

In [15]:
planner = GAMealPlanner(meal_planning_environment)

In [16]:
NUM_GENERATIONS = 10
NUM_PARENTS_MATING = 20
POPULATION_SIZE = 100
NUM_DAYS = 7
NUM_MEALS_PER_DAY = 3
GENERATION_PRINT_INTERVAL = 10
SEED = 1

In [17]:
best_meal_plan, best_fitness_score = planner.generate_meal_plan(
    num_days=NUM_DAYS,
    meals_per_day=NUM_MEALS_PER_DAY,
    num_generations=NUM_GENERATIONS,
    num_parents_mating=NUM_PARENTS_MATING,
    population_size=POPULATION_SIZE,
    generation_print_interval=GENERATION_PRINT_INTERVAL,
    seed=SEED,
)

Generation 10, Best Fitness: -56.52


In [18]:
print(f"Best fitness score: {best_fitness_score:.2f}")

Best fitness score: -56.52


In [19]:
pantry_consumption_df = planner.get_pantry_consumption()
pantry_consumption_df

,Ingredient,Available,Consumed,Unused,Expires in
0,chicken breast,800,0,800,1d
1,broccoli,1500,0,1500,3d
2,rice,1000,0,1000,4d


In [20]:
meal_plan_df = planner.get_meal_plan_recipes()
meal_plan_df

,Day,Meal 1,Meal 2,Meal 3
0,1,Artichoke and Fennel Ravioli with Tomato-Fenne...,Stout and Cheddar Rarebit with Fried Eggs,Bistro Steak with Buttermilk Onion Rings
1,2,Bolognese Sauce,Bee's Knees,"Grilled Swiss Cheese, Tuna and Red Pepper Sand..."
2,3,Roast Turkey with White-Wine Gravy,Whipped Crème Fraîche,Wagon-Wheel Pasta & Goat Cheese
3,4,Small Semolina Griddle Breads,Tropical Fruit and Rum Punch,Zucchini Salad With Ajo Blanco Dressing & Spic...
4,5,Peppered Lamb with Pine Nut Sauce,Butternut Squash with Shallots and Sage,Chocolate Mint Chip Parfait
5,6,Chocolate Curls,"Pork Loin with Apples, Prunes, and Mustard Cre...",Bourbon-Orange Pecan Pie with Bourbon Cream
6,7,Prairie Oyster I,Bacon Baklava,Pan Bagnat


In [21]:
shopping_list_df, num_ingredients, total_cost = planner.get_shopping_list()

print(f"Total ingredients needed to purchase: {num_ingredients}")
print(f"Total cost: €{total_cost:.2f}")

shopping_list_df

Total ingredients needed to purchase: 152
Total cost: €48.56


,Ingredient,Quantity to Buy (g),Cost (€)
0,1Butter Pie Crust Dough disk,16.7,0.17
1,Dijon mustard,2.5,0.02
2,Fresh fennel fronds,2.0,0.02
3,Fresh pineapple,2.2,0.02
4,Granny Smith apples,50.0,0.50
...,...,...,...
148,white bread,49.0,0.49
149,whole egg,14.2,0.14
150,whole milk,12.5,0.12
151,zucchinis,24.0,0.24


In [22]:
daily_nutrition_df = planner.get_daily_nutrition()
daily_nutrition_df

,Calories,Protein,Δ Calories and Target Calories,Δ Protein and Target Protein
Day 1,2161.5 kcal,37.3 g,161.5 kcal,-12.7 g
Day 2,1065.9 kcal,65.3 g,-934.1 kcal,15.3 g
Day 3,1933.2 kcal,19.3 g,-66.8 kcal,-30.7 g
Day 4,2053.4 kcal,21.3 g,53.4 kcal,-28.7 g
Day 5,1428.7 kcal,53.7 g,-571.3 kcal,3.7 g
Day 6,1785.7 kcal,21.2 g,-214.3 kcal,-28.8 g
Day 7,1390.9 kcal,63.2 g,-609.1 kcal,13.2 g
